# 69 · PT pathways without mitochondrial reads in the offset

**The question.** Every NB gene model so far uses offset log(library). The library is the total over all
orthologs measured in the dataset, and it includes the mitochondrially encoded genes. Their share of the library
changes along the PT, and differently by species:
- on SCF13, from the first to the last fifth of the coordinate, mt reads fall from 21.5% to 12.4% of the human
  library and rise from 3.4% to 5.5% of the mouse library;
- the 61 nuclear-encoded OXPHOS subunits stay flat, so this is not mitochondrial content.

The offset therefore tilts every gene's human/mouse log ratio toward S3, by about 0.13 natural-log units. Genes that
are not significant drift too, so the calls lean toward genes "rising in human toward S3".

This notebook reruns the two pathway methods with **exactly one change**: the mitochondrially encoded genes
(mouse symbol prefix `mt-`) leave the library and the tested universe.
- **Normalized:** notebook 67's primary, cameraPR on the z of `T_spatial` normalized by specimen shape variation.
- **Clustering:** notebook 66's clustering-first ORA from the `pseudospace_reconstruction` package.

**The frozen protocol.** `docs/results/pt-gene-model-nb.md`, section "Notebook 69 protocol: pathway tests without
mitochondrial reads in the offset", committed in 4fdcdf4 before any fit. Its checks:
- **(a) Null drift.** The median over non-significant genes of d_g = mean Δ over the last 20% of the grid minus the
  first 20%. Criterion (human vs mouse): |after| ≤ 0.05 natural-log units.
- **(b) Direction balance** of the significant genes (descriptive).
- **(c) Calls and pass rules** of both methods on both datasets.
- **(d) Overlap** between the methods, with and without the 8 proteasome subunits.
- **(e) Lost and gained calls** against notebooks 66 and 67.
- **Decision:** if (a) passes, these calls replace notebook 66/67's as the primary for both methods.

**Two datasets**, built exactly as in notebooks 66 and 67, minus the mt genes:

| Dataset | Structures | Genes (66/67 → 69) | Pathways | Reference | Case | Coordinate |
|---|---|---|---|---|---|---|
| Human vs mouse PT | 12,272 | 11,071 → 11,059 | 1,513 | 2 control mice | 2 cortex sections of 1 human donor | SCF13 |
| AKI vs control mouse PT | 16,991 | 9,107 → 9,095 | 1,429 | 2 control mice | 2 AKI mice | mouse-only PT DPT |

**Sections:**
1. Setup, inputs and guards.
2. The gene model and the specimen-shape denominator.
3. The normalized method.
4. The clustering method.
5. Check (c): calls and pass rules.
6. Checks (a) and (b): null drift and direction balance.
7. Check (d): overlap between the methods.
8. Check (e): lost and gained calls.
9. The figure.
10. Readings of the protocol.
11. Summary.
12. Run record.

**Assumptions.** Each coordinate is treated as exact. Specimens, not structures, are the replicates. The two human
"replicates" are two sections of one donor. With two specimens per group every result is descriptive.

## 1 · Setup

In [ ]:
import argparse
import json
import os
import sys
import time
import warnings
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
coordinate_file = results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv'
upstream13 = results_root / 'minimal_pt_scfates'
mouse_v5 = results_root / 'mouse_only_v5'
universe_file = results_root / 'pt_pathway_remodeling_scfates' / 'gene_statistics.csv'      # notebook 45's universe
nb66 = results_root / 'pt_pathway_pipelines_compared'    # read only: stage keys, caches, call sets, zones, clusters
nb67 = results_root / 'pt_normalized_t_spatial'          # read only: stage keys, caches, gene statistics, tests
output = results_root / 'pt_pathways_without_mito'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)
DATASETS = ['human_vs_mouse', 'aki_vs_control']
LIBRARIES = ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')
REQUIRED = [coordinate_file, upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            universe_file, mouse_v5 / 'all_mouse_tubules_scanpy_dpt.h5ad',
            nb66 / 'tables' / 'method_table.csv', nb66 / 'tables' / 'method_b_zones.csv', nb67 / 'tables' / 'method_table.csv',
            *[nb66 / 'tables' / f'{kind}_{ds}.csv' for kind in ('redundancy_clusters', 'pathway_detail') for ds in DATASETS],
            *[nb67 / 'tables' / f'gene_statistics_{ds}_primary.csv' for ds in DATASETS],
            *[data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json' for lib in LIBRARIES]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 13, 36, 66 and 67 and the mouse_only_v5 workflow first)')

NOTEBOOK_LOGIC_VERSION = '69.no_mito.1'   # Bump when a cached calculation changes.
MT_PREFIX = 'mt-'                                # mitochondrially encoded genes (mouse symbols)
PROTEASOME = ['Psma3', 'Psma4', 'Psmb5', 'Psmb6', 'Psmc2', 'Psmc6', 'Psmd11', 'Psmd14']   # check (d) sensitivity
ALPHA, NCDR_MAX, NULL_RATE = .05, .25, .05      # discovery level and notebook 31's relabeling rule
CALIBRATION_MAX = 1.5 * ALPHA                    # notebook 61's criterion 2: share of relabeled p <= 0.05
CAMERA_COR = .01                                 # limma's default inter.gene.cor (notebook 67)
BASIS_DF, GRID_POINTS = 6, 101                   # the shared gene model
DF1 = 6                                          # numerator: group x basis
DF2 = {'condition': 12, 'relabeling': 6}         # denominator: specimen-within-group x basis (asserted)
GENE_FDR = .05                                   # check (b): significant genes on the normalized F
NULL_Q = .5                                      # check (a): genes with q > 0.5 in both runs
DRIFT_SHARE = .2                                 # d_g: mean delta over the last 20% of the grid minus the first 20%
DRIFT_LIMIT = .05                                # check (a): |after| <= 0.05 natural-log units (human vs mouse)
SET_SIZES = (10, 300)
REDUNDANCY_JACCARD = .5                          # notebook 66's redundancy rule
ZONE_MARKERS = {'PT-S1': ['Slc5a12'], 'PT-S2': ['Slc13a3'], 'PT-S3': ['Slc16a9']}   # notebook 66
ZONE_SHORT = {'PT-S1': 'S1', 'PT-S2': 'S2', 'PT-S3': 'S3'}
NODE_SPACING = 1 / 29                            # notebook 66
EXTERNAL_COMMIT = '59da04b'                      # pseudospace_reconstruction, as in notebook 66
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '36'))   # parallel workers; never part of a cache key

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from importlib.metadata import distribution, version
from IPython.display import display

direct_url = json.loads(distribution('pseudospace-reconstruction').read_text('direct_url.json') or '{}')
EXTERNAL = {'version': version('pseudospace-reconstruction'),
            'commit': direct_url.get('vcs_info', {}).get('commit_id', 'unknown'), 'url': direct_url.get('url', '')}
assert EXTERNAL['commit'].startswith(EXTERNAL_COMMIT), f'pseudospace_reconstruction is not commit {EXTERNAL_COMMIT}: {EXTERNAL}'

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
N_COLOR, B_COLOR = '#1b8a5a', '#eb6834'          # notebook 67's normalized and notebook 66's clustering colours
BEFORE_COLOR, AFTER_COLOR = MUTED, N_COLOR       # before: dashed muted ink; after: solid green (validated pair)
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


print('Results folder:', output, '· parallel workers:', N_JOBS, '· pseudospace_reconstruction', EXTERNAL['version'],
      EXTERNAL['commit'][:7])

### 1.1 · Inputs, with and without the mitochondrially encoded genes

**The mt genes** are the ortholog-map mouse symbols that start with `mt-`: 13 names.
- **Human vs mouse.** `pt_inputs(..., exclude_genes=MT)` removes them from the library (the total over the
  orthologs measured in both inputs) and from the universe. mt-Atp8 is not measured in human, so 12 genes leave.
- **AKI vs control.** Notebook 66's construction (its cell 4, mirroring notebook 45), with the library summed over
  the measured panel genes except the 13 mt genes. The universe loses the 12 that are in notebook 12's universe.
- The pathways are rebuilt on the reduced universe with the same 10–300 rule.
- The log-normalized matrix keeps its full-library normalization. It feeds only the zone cuts and nothing in either
  method's calls (protocol).

The inputs with the mt genes kept are built too, for the guards only, and then dropped.

**Guards (not cached):**
- exactly 12 (human vs mouse) and 12 (AKI) mt genes leave the universe; 12 and 13 leave the library, and the
  library falls by exactly their counts;
- the inputs with the mt genes kept reproduce notebook 66's and 67's recorded gene-model stage keys (counts digest,
  library, position, specimen, genes, group, nuisance), recomputed with the code digest in their cache sidecars;
- structures, positions, specimens, segments and the non-mt counts are unchanged;
- the genes are notebook 67's minus the mt genes; the pathways equal notebook 66's (ids and sizes).

In [ ]:
import anndata as ad
from scipy import sparse

from pseudospace.coordinate_inputs import pt_inputs, read_coordinate
from pseudospace.pathway_pipelines import partition_groups
from pseudospace.pathways import build_pathway_membership
from pseudospace.stage_cache import cached_payload, digest, stage_key

t0 = time.perf_counter()
coordinate = read_coordinate(coordinate_file)
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith(MT_PREFIX))
assert len(MT) == 13, MT
DATA, KEPT, LIBRARY_ROWS = {}, {}, []

# Human vs mouse: notebook 37's construction on SCF13 (pt_inputs), as notebooks 66 and 67
kept = pt_inputs(results_root, data_root, coordinate)                       # notebook 66/67's inputs (guards only)
hm = pt_inputs(results_root, data_root, coordinate, exclude_genes=MT)
assert len(kept['position']) == 12272 and len(kept['genes']) == 11071 and len(kept['gene_sets']) == 1513, "notebook 37's universe changed"
mt_universe = [g for g in kept['genes'] if g in MT]
assert len(mt_universe) == 12 and 'mt-Atp8' not in mt_universe, mt_universe
assert hm['genes'] == [g for g in kept['genes'] if g not in MT] and len(hm['genes']) == 11059
index = pd.Index(kept['genes'])
mt_counts = np.asarray(kept['counts'][:, index.get_indexer(mt_universe)].sum(axis=1)).ravel()
np.testing.assert_allclose(hm['library'], kept['library'] - mt_counts, rtol=0, atol=1e-6)   # only the 12 genes leave
assert (hm['counts'].tocsr() != kept['counts'].tocsr()[:, index.get_indexer(hm['genes'])]).nnz == 0
for key in ('position', 'human', 'specimen', 'segment'):
    np.testing.assert_array_equal(hm[key], kept[key])
assert hm['gene_sets'] == kept['gene_sets'], 'the pathways changed on the reduced universe'
DATA['human_vs_mouse'] = {
    'title': 'Human vs mouse PT', 'case_name': 'human', 'reference_name': 'mouse', 'coordinate': 'SCF13',
    'counts': hm['counts'].tocsr(), 'library': hm['library'], 'library_with_mt': kept['library'],
    'position': hm['position'], 'specimen': hm['specimen'].astype(str),
    'segment': pd.Series(hm['segment']).str.replace('PT-', '', regex=False).to_numpy(),
    'case': hm['human'].astype(bool), 'genes': list(hm['genes']), 'sets': hm['gene_sets'], 'lognorm': hm['Y'].tocsr(),
    'mt_universe': mt_universe, 'mt_library': mt_universe}
KEPT['human_vs_mouse'] = {'key': {'counts': digest(kept['counts'].tocsr()), 'library': kept['library'],
                                  'position': kept['position'], 'specimen': kept['specimen'].astype(str),
                                  'genes': list(kept['genes'])},
                          'genes': list(kept['genes']), 'sets': kept['gene_sets']}
del hm, kept

# AKI vs control: notebook 66's cell 4 (notebook 45's cells 4, 8 and 10), with the mt genes out of library and universe
universe_genes = pd.read_csv(universe_file, index_col=0).index.tolist()


def pathway_sets(universe):
    """Notebook 37's pathways on a gene universe: members among the universe, 10-300 of them (notebook 66's rule)."""
    coverage = pd.concat([build_pathway_membership(
        json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json').read_text()), universe,
        ortholog_map=orthologs, library_name=lib, tested=universe, min_genes=1) for lib in LIBRARIES], ignore_index=True)
    coverage['pathway_id'] = coverage.library + '::' + coverage.pathway
    coverage = coverage[coverage.n_tested.between(SET_SIZES[0], min(SET_SIZES[1], len(universe) - 1))]
    return {row.pathway_id: list(row.genes_present) for row in coverage.itertuples()}


def restrict(sets, genes):
    keep = set(genes)
    sets = {p: [g for g in v if g in keep] for p, v in sets.items()}
    return {p: v for p, v in sets.items() if SET_SIZES[0] <= len(v) <= min(SET_SIZES[1], len(keep) - 1)}


gene_sets_kept = pathway_sets(universe_genes)
gene_sets_all = pathway_sets([g for g in universe_genes if g not in MT])    # rebuilt on the reduced universe
assert len(gene_sets_kept) == 1513 and gene_sets_all == gene_sets_kept, "notebook 37's pathways changed"

v5 = ad.read_h5ad(mouse_v5 / 'all_mouse_tubules_scanpy_dpt.h5ad')
v5 = v5[v5.obs.broad_tubule_marker_call.astype(str).eq('PT').to_numpy()
        & v5.obs.segment_class.astype(str).isin(['PT-S1', 'PT-S2', 'PT-S3']).to_numpy()].copy()
support = v5.obs.groupby('sample', observed=True).pt_subset_scanpy_dpt.quantile([.01, .99]).unstack()
position_v5 = v5.obs.pt_subset_scanpy_dpt.to_numpy(float)
v5 = v5[(position_v5 >= support[.01].max()) & (position_v5 <= support[.99].min())].copy()
aki_position = v5.obs.pt_subset_scanpy_dpt.to_numpy(float)
aki_position = (aki_position - aki_position.min()) / (aki_position.max() - aki_position.min())
panel = np.isin(v5.var_names, list(set(orthologs.mouse_symbol)))
aki_raw = sparse.csr_matrix(v5.layers['counts'])[:, panel]
aki_var = v5.var_names[panel]
aki_specimen = v5.obs['sample'].astype(str).to_numpy()
aki_segment = v5.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
del v5
measured = sorted(set(aki_var[np.asarray(aki_raw.sum(axis=0)).ravel() > 0]))
aki_counts_all = sparse.csr_matrix(aki_raw[:, pd.Index(aki_var).get_indexer(measured)], dtype=float)
aki_library_kept = np.asarray(aki_counts_all.sum(axis=1)).ravel()
mt_library = [g for g in measured if g in MT]
assert len(mt_library) == 13, mt_library
aki_library = aki_library_kept - np.asarray(aki_counts_all[:, pd.Index(measured).get_indexer(mt_library)].sum(axis=1)).ravel()
aki_lognorm_all = aki_counts_all.multiply((1e4 / aki_library_kept)[:, None]).tocsr()   # unchanged normalization
aki_lognorm_all.data = np.log1p(aki_lognorm_all.data)
pool_genes = [g for g in universe_genes if g in set(measured)]
x = aki_lognorm_all[:, pd.Index(measured).get_indexer(pool_genes)]
per_donor = {}
for name in np.unique(aki_specimen):
    rows, seg = x[aki_specimen == name], aki_segment[aki_specimen == name]
    present = np.array([np.asarray((rows[seg == s] > 0).sum(axis=0)).ravel() > 0 for s in np.unique(seg)])
    per_donor[name] = (np.asarray(rows.mean(axis=0)).ravel(), np.asarray((rows > 0).mean(axis=0)).ravel(), present.mean(axis=0))
aki_covariates = pd.DataFrame({key: np.mean([per_donor[d][k] for d in per_donor], axis=0)
                               for k, key in enumerate(('mean_expression', 'detection', 'coverage'))}, index=pool_genes)
aki_genes_kept = list(aki_covariates.index[aki_covariates.detection.ge(.02)])
aki_genes = [g for g in aki_genes_kept if g not in MT]
mt_universe = [g for g in aki_genes_kept if g in MT]
assert len(aki_position) == 16991 and len(aki_genes_kept) == 9107 and len(aki_genes) == 9095 and len(mt_universe) == 12
aki_sets_kept, aki_sets = restrict(gene_sets_kept, aki_genes_kept), restrict(gene_sets_all, aki_genes)
assert len(aki_sets_kept) == 1429 and aki_sets == aki_sets_kept, "notebook 45's AKI pathways changed"
kept_columns, aki_columns = pd.Index(measured).get_indexer(aki_genes_kept), pd.Index(measured).get_indexer(aki_genes)
DATA['aki_vs_control'] = {
    'title': 'AKI vs control mouse PT', 'case_name': 'AKI', 'reference_name': 'control', 'coordinate': 'mouse-only PT DPT',
    'counts': aki_counts_all[:, aki_columns].tocsr(), 'library': aki_library, 'library_with_mt': aki_library_kept,
    'position': aki_position, 'specimen': aki_specimen, 'segment': aki_segment,
    'case': np.isin(aki_specimen, ['IR2A2', 'IR2A4']), 'genes': aki_genes, 'sets': aki_sets,
    'lognorm': aki_lognorm_all[:, aki_columns].tocsr(), 'mt_universe': mt_universe, 'mt_library': mt_library}
KEPT['aki_vs_control'] = {'key': {'counts': digest(aki_counts_all[:, kept_columns].tocsr()), 'library': aki_library_kept,
                                  'position': aki_position, 'specimen': aki_specimen, 'genes': aki_genes_kept},
                          'genes': aki_genes_kept, 'sets': aki_sets_kept}
del aki_raw, aki_counts_all, aki_lognorm_all, x
assert list(DATA) == DATASETS

PARTITIONS, SHORT = {}, {}


def group_one(ds, label):
    """The specimens in group 1 of a partition (the case specimens for the condition split)."""
    d = DATA[ds]
    return sorted(set(d['specimen'][PARTITIONS[ds][label][0] == 1]))


def newest_sidecar(folder, stage):
    """Metadata of a notebook's newest cache entry of one stage (read only), or None."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz.key.json'),
                   key=lambda p: json.loads(p.read_text())['written_utc'])
    return json.loads(files[-1].read_text()) if files else None


def slug(label):
    return label.replace(':', '_').replace('+', '_')


guard_log = []
for ds, d in DATA.items():
    assert d['counts'].shape == d['lognorm'].shape == (len(d['position']), len(d['genes']))
    assert all(set(v) <= set(d['genes']) for v in d['sets'].values())
    assert (d['library'] > 0).all()
    PARTITIONS[ds] = partition_groups(d['specimen'], d['case'])
    labels = list(PARTITIONS[ds])
    assert labels[0] == 'species'
    SHORT[ds] = dict(zip(labels, ['condition', 'relabeling 1', 'relabeling 2']))
    d['input_key'] = {'counts': digest(d['counts']), 'library': d['library'], 'position': d['position'],
                      'specimen': d['specimen'], 'genes': d['genes']}
    d['log_mean_count'] = np.log(np.asarray(d['counts'].mean(axis=0)).ravel())    # notebook 67's squeezeVar covariate
    d['case_specimens'] = sorted(set(d['specimen'][d['case']]))
    d['reference_specimens'] = sorted(set(d['specimen'][~d['case']]))
    # Guard: with the mt genes kept, the inputs are notebook 66's and 67's (their recorded gene-model stage keys)
    group, nuisance = PARTITIONS[ds]['species']
    for folder in (nb66, nb67):
        meta = newest_sidecar(folder, f'gene_model_{ds}_species')
        assert meta is not None, f'{folder.name}: no cached gene model to check the inputs against'
        key = stage_key(meta['stage'], params={'basis_df': BASIS_DF, 'grid_points': GRID_POINTS, 'label': 'species'},
                        inputs={**KEPT[ds]['key'], 'group': group, 'nuisance': nuisance, 'alpha': None},
                        code=meta['code_digest'])
        assert key == meta['key'], (ds, folder.name, key, meta['key'])
    # Guard: genes = notebook 67's minus mt; pathways = notebook 66's (ids and sizes)
    genes67 = pd.read_csv(nb67 / 'tables' / f'gene_statistics_{ds}_primary.csv', header=[0, 1], index_col=0).index
    assert list(genes67) == KEPT[ds]['genes'] and d['genes'] == [g for g in genes67 if g not in MT]
    detail66 = pd.read_csv(nb66 / 'tables' / f'pathway_detail_{ds}.csv', index_col='pathway_id')
    assert list(detail66.index) == list(d['sets'])
    assert (detail66['n genes'].to_numpy() == np.array([len(v) for v in d['sets'].values()])).all()
    LIBRARY_ROWS.append({'dataset': d['title'], 'mt genes out of the library': len(d['mt_library']),
                         'mt genes out of the universe': len(d['mt_universe']),
                         'mt share of the old library (all structures)': 1 - d['library'].sum() / d['library_with_mt'].sum(),
                         'genes (66/67 → 69)': f"{len(KEPT[ds]['genes']):,} → {len(d['genes']):,}",
                         'pathways': len(d['sets'])})
    guard_log.append(f"{d['title']}: {len(d['mt_library'])} mt genes out of the library and {len(d['mt_universe'])} out of "
                     f"the universe; with them kept the inputs reproduce notebook 66's and 67's gene-model stage keys; "
                     f"genes = notebook 67's minus mt; pathways = notebook 66's")

INPUT_TABLE = pd.DataFrame({d['title']: {
    'structures': len(d['position']), 'genes': len(d['genes']), 'pathways': len(d['sets']),
    'reference specimens': ', '.join(f"{s} ({int((d['specimen'] == s).sum()):,})" for s in d['reference_specimens']),
    'case specimens': ', '.join(f"{s} ({int((d['specimen'] == s).sum()):,})" for s in d['case_specimens']),
    **{f'{SHORT[ds][label]} (one side)': ' + '.join(group_one(ds, label)) for label in list(PARTITIONS[ds])[1:]},
    'mt genes out of the library': ', '.join(d['mt_library']), 'mt genes out of the universe': len(d['mt_universe'])}
    for ds, d in DATA.items()})
INPUT_TABLE.to_csv(tables / 'inputs.csv')
display(INPUT_TABLE)
display(pd.DataFrame(LIBRARY_ROWS).set_index('dataset').round(4))
print('\n'.join('· ' + line for line in guard_log))
print(f'Inputs {time.perf_counter() - t0:.0f} s')

### 1.2 · Redundancy clusters

Notebook 66's rule (the package's `cluster_gene_sets`: 1 − Jaccard of membership within the tested genes, average
linkage, cut at mean Jaccard 0.5, each cluster named by its medoid), recomputed on the reduced universe. **Guard:**
the clusters equal notebook 66's.

In [ ]:
from pseudospace_reconstruction.compare.pathways import cluster_gene_sets
from pseudospace_reconstruction.io.gene_sets import build_gene_sets

EXT_SETS, REDUNDANCY = {}, {}
for ds, d in DATA.items():
    sets = build_gene_sets({lib: {p.split('::', 1)[1]: v for p, v in d['sets'].items() if p.startswith(lib + '::')}
                            for lib in LIBRARIES}, d['genes'], min_genes=SET_SIZES[0], max_genes=SET_SIZES[1])
    ids = [f'{lib}::{name}' for lib, name in zip(sets.library, sets.pathway)]
    assert sorted(ids) == sorted(d['sets']), "the package's gene-set builder and ours disagree on the tested pathways"
    assert all(set(sets.member_names(i)) == set(d['sets'][p]) for i, p in enumerate(ids)), 'members differ'
    EXT_SETS[ds] = sets
    clusters = cluster_gene_sets(sets.membership(), sets.names, REDUNDANCY_JACCARD)
    clusters.index = pd.Index(ids, name='pathway_id')
    medoid = clusters[clusters.is_representative].reset_index().set_index('cluster_id').pathway_id
    clusters['representative'] = clusters.cluster_id.map(medoid)
    REDUNDANCY[ds] = clusters.reindex(list(d['sets']))
    old = pd.read_csv(nb66 / 'tables' / f'redundancy_clusters_{ds}.csv', index_col='pathway_id').reindex(list(d['sets']))
    assert (REDUNDANCY[ds].cluster_id.astype(str).to_numpy() == old.cluster_id.astype(str).to_numpy()).all()
    assert (REDUNDANCY[ds].representative.astype(str).to_numpy() == old.representative.astype(str).to_numpy()).all()
    REDUNDANCY[ds].to_csv(tables / f'redundancy_clusters_{ds}.csv')
guard_log.append("redundancy clusters on the reduced universe = notebook 66's, both datasets")
print({DATA[ds]['title']: f"{len(DATA[ds]['sets']):,} pathways in {REDUNDANCY[ds].cluster_id.nunique():,} redundancy clusters"
       for ds in DATASETS})


def representatives(ds, calls):
    """The redundancy clusters a call set touches, named by their medoids."""
    return set(REDUNDANCY[ds].representative.reindex(sorted(calls)))


NAMES = {ds: pd.Series({p: p.split('::', 1)[1] for p in DATA[ds]['sets']}).str.replace(r' R-HSA-\d+$', '', regex=True)
         for ds in DATASETS}

## 2 · The gene model and the specimen-shape denominator

Notebook 67's section 2, unchanged except for the library:
- **Gene model** (`standard_gene_model`): an NB GLM on raw counts with offset log(library), now without the mt
  reads; a 6-df cubic B-spline with knots at the position quartiles; prior weights wᵢ = n / (2 · J · n_j); the
  101-point grid. α is re-estimated per gene under the condition split's M_full with the new offset and reused for
  both relabelings, where the condition's own shape enters as a nuisance term.
- **Denominator** (`specimen_shape_lr`): LR_spec = D(M_full) − D(M_spec), with the same α, weights and offset.

**Guards:** df₂ = 12 for the condition split and 6 for each relabeling, from the design ranks; the specimen curves
are identical in all three partitions; the grid equals notebook 67's (it depends on positions only).

In [ ]:
from pseudospace import nb_gam, pathway_calibration, pathway_pipelines, stats_gam
from pseudospace.pathway_pipelines import GeneModel, specimen_shape_lr, standard_gene_model

cache = output / 'stage_cache'
code_key = digest([NOTEBOOK_LOGIC_VERSION] + [Path(m.__file__) for m in (nb_gam, pathway_calibration, pathway_pipelines, stats_gam)])
SECONDS = {}   # uncached compute time of every cached stage


def cached(stage, compute, *, params=None, inputs=None, key=None):
    """cached_payload under this notebook's root and code; keeps the stage's uncached compute time."""
    def wrapped():
        t = time.perf_counter()
        out = dict(compute())
        out['__seconds'] = np.array(time.perf_counter() - t)
        return out
    result = cached_payload(stage, wrapped, root=cache, params=params, inputs=inputs, code=key or code_key)
    SECONDS[stage] = float(result.pop('__seconds'))
    return result


def model_payload(model):
    out = {f'stat|{c}': model.statistics[c].to_numpy() for c in model.statistics}
    out.update(grid=model.grid, knots=model.knots, reference=model.reference, case=model.case, delta=model.delta,
               se=model.se, group_names=model.groups.index.to_numpy().astype(str), group_values=model.groups.to_numpy(),
               alpha=np.asarray(model.dispersion['alpha'], float), alpha_converged=np.asarray(model.dispersion['converged'], bool))
    for name, curve in model.specimen_curves.items():
        out[f'curve|{name}'] = curve
        out[f'converged|{name}'] = model.specimen_converged[name].to_numpy()
        out[f'separated|{name}'] = model.specimen_separated[name].to_numpy()
    return out


def model_from(payload, genes):
    index = pd.Index(genes, name='gene')
    stats = pd.DataFrame({c: payload[f'stat|{c}'] for c in ('T_level', 'T_spatial', 'T_total', 'alpha', 'converged',
                                                            'separated')}, index=index)
    stats[['converged', 'separated']] = stats[['converged', 'separated']].astype(bool)
    names = sorted(k.split('|', 1)[1] for k in payload if k.startswith('curve|'))
    return GeneModel(statistics=stats, grid=payload['grid'], knots=payload['knots'], reference=payload['reference'],
                     case=payload['case'], delta=payload['delta'], se=payload['se'],
                     specimen_curves={n: payload[f'curve|{n}'] for n in names},
                     specimen_converged=pd.DataFrame({n: payload[f'converged|{n}'].astype(bool) for n in names}, index=index),
                     specimen_separated=pd.DataFrame({n: payload[f'separated|{n}'].astype(bool) for n in names}, index=index),
                     groups=pd.Series(payload['group_values'].astype(int), index=payload['group_names'].astype(str)),
                     dispersion={'alpha': payload['alpha'], 'converged': payload['alpha_converged'].astype(bool)})


def stored(folder, stage):
    """A notebook's newest cached payload of one stage (read only); raises when its cache was purged."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}; the protocol takes the "before" values from it')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


MODELS, DENOM = {}, {}
for ds, d in DATA.items():
    MODELS[ds], DENOM[ds] = {}, {}
    for label, (group, nuisance) in PARTITIONS[ds].items():
        dispersion = None if label == 'species' else MODELS[ds]['species'].dispersion
        payload = cached(f'gene_model_{ds}_{slug(label)}', lambda d=d, group=group, nuisance=nuisance, dispersion=dispersion:
                         model_payload(standard_gene_model(d['counts'], d['library'], d['position'], group, d['specimen'],
                                                           genes=d['genes'], nuisance_shape=nuisance, dispersion=dispersion,
                                                           basis_df=BASIS_DF, n_jobs=N_JOBS)),
                         params={'basis_df': BASIS_DF, 'grid_points': GRID_POINTS, 'label': label},
                         inputs={**d['input_key'], 'group': group, 'nuisance': nuisance,
                                 'alpha': None if dispersion is None else dispersion['alpha']})
        MODELS[ds][label] = model_from(payload, d['genes'])
        alpha = MODELS[ds]['species'].dispersion
        DENOM[ds][label] = cached(f'specimen_shape_{ds}_{slug(label)}',
                                  lambda d=d, group=group, nuisance=nuisance, alpha=alpha:
                                  specimen_shape_lr(d['counts'], d['library'], d['position'], group, d['specimen'], alpha,
                                                    nuisance_shape=nuisance, basis_df=BASIS_DF, n_jobs=N_JOBS),
                                  params={'basis_df': BASIS_DF, 'label': label},
                                  inputs={**d['input_key'], 'group': group, 'nuisance': nuisance, 'alpha': alpha['alpha'],
                                          'alpha_converged': alpha['converged']})

BEFORE_MODEL = {ds: stored(nb67, f'gene_model_{ds}_species') for ds in DATASETS}   # notebook 67's (= 66's) condition split
for ds in DATASETS:
    condition = MODELS[ds]['species']
    assert len(condition.grid) == GRID_POINTS
    np.testing.assert_array_equal(condition.grid, BEFORE_MODEL[ds]['grid'])
    for label, model in MODELS[ds].items():
        expected = DF2['condition' if label == 'species' else 'relabeling']
        assert int(DENOM[ds][label]['df2']) == expected, (ds, label, int(DENOM[ds][label]['df2']))
        np.testing.assert_array_equal(model.grid, condition.grid)
        for name, curve in model.specimen_curves.items():
            np.testing.assert_allclose(curve, condition.specimen_curves[name], rtol=1e-9, atol=1e-9)
guard_log.append(f"df2 = {DF2['condition']} (condition split) and {DF2['relabeling']} (each relabeling); specimen curves "
                 "identical in all three partitions; grid = notebook 67's, both datasets")

MODEL_ROWS = []
for ds in DATASETS:
    before_alpha = BEFORE_MODEL[ds]['alpha']
    keep = pd.Index(KEPT[ds]['genes']).get_indexer(DATA[ds]['genes'])
    for label, model in MODELS[ds].items():
        den = DENOM[ds][label]
        MODEL_ROWS.append({
            'dataset': DATA[ds]['title'], 'partition': SHORT[ds][label], 'median α': float(np.median(model.statistics.alpha)),
            'median α, notebook 67': float(np.median(before_alpha[keep])) if label == 'species' else np.nan,
            'non-converged (median-filled)': int((~model.statistics.converged).sum()),
            'separated (flagged)': int(model.statistics.separated.sum()), 'median T_spatial': float(model.statistics.T_spatial.median()),
            'df2': int(den['df2']), 'median LR_spec / df2': float(np.median(np.asarray(den['LR_spec'], float) / int(den['df2']))),
            'gene model seconds': round(SECONDS[f'gene_model_{ds}_{slug(label)}'], 1),
            'denominator seconds': round(SECONDS[f'specimen_shape_{ds}_{slug(label)}'], 1)})
MODEL_TABLE = pd.DataFrame(MODEL_ROWS)
MODEL_TABLE.to_csv(tables / 'gene_model_summary.csv', index=False)
display(MODEL_TABLE.round(3))
print('\n'.join('· ' + line for line in guard_log))

## 3 · The normalized method

Notebook 67's primary, unchanged: s² = LR_spec / df₂ moderated by `limma::squeezeVar` with the log mean raw count per
structure as covariate; F = (`T_spatial` / 6) / s²_post with df (6, df₂ + d₀); z = Φ⁻¹(1 − p)
(`pathway_pipelines.moderated_f`). The gene-level q is the BH of p over all tested genes.

Each pathway is tested by `limma::cameraPR` on z (`inter.gene.cor` 0.01, `use.ranks` FALSE). A call is Up with
FDR ≤ 0.05. For calibration the one-sided p is p/2 when Up, else 1 − p/2.

In [ ]:
from scipy.stats import false_discovery_control

from pseudospace.pathway_pipelines import camera_pr, moderated_f

STAT, TESTS_N, N_CALLS = {ds: {} for ds in DATASETS}, {ds: {} for ds in DATASETS}, {ds: {} for ds in DATASETS}
SQUEEZE_ROWS = []
for ds, d in DATA.items():
    for label, model in MODELS[ds].items():
        den = DENOM[ds][label]
        df2 = int(den['df2'])
        frame = moderated_f(model.statistics.T_spatial, DF1, np.asarray(den['LR_spec'], float), df2,
                            covariate=d['log_mean_count'], robust=False)
        frame['q'] = false_discovery_control(frame.p.to_numpy())
        frame['converged'] = model.statistics.converged.to_numpy() & np.asarray(den['converged'], bool)
        frame['separated'] = model.statistics.separated.to_numpy() | np.asarray(den['separated'], bool)
        assert np.isfinite(frame.z).all() and np.isfinite(frame.F).all()
        STAT[ds][label] = frame
        test = camera_pr(frame.z, d['sets'], inter_gene_cor=CAMERA_COR, use_ranks=False, fdr=ALPHA)
        assert list(test.pathway_id) == list(d['sets'])
        TESTS_N[ds][label] = test
        N_CALLS[ds][label] = set(test.pathway_id[test.call.astype(bool)])
        test.assign(partition=label).to_csv(tables / f'pathway_tests_{ds}_cameraPR_{slug(label)}.csv', index=False)
        SQUEEZE_ROWS.append({'dataset': d['title'], 'partition': SHORT[ds][label], 'df2': df2,
                             'd0 (median)': float(frame.d0.median()), 's0² (median)': float(frame.s2_prior.median()),
                             'median F': float(frame.F.median()), 'genes p ≤ 0.05': int((frame.p <= .05).sum()),
                             'genes BH ≤ 0.05': int((frame.q <= GENE_FDR).sum()), 'pathway calls': len(N_CALLS[ds][label])})
    pd.concat({SHORT[ds][label]: STAT[ds][label] for label in PARTITIONS[ds]}, axis=1).to_csv(tables / f'gene_statistics_{ds}.csv')
SQUEEZE = pd.DataFrame(SQUEEZE_ROWS)
SQUEEZE.to_csv(tables / 'moderation.csv', index=False)
display(SQUEEZE.set_index(['dataset', 'partition']).round(3))

## 4 · The clustering method

Notebook 66's method B, unchanged: the `pseudospace_reconstruction` package's stage 06 (`compare_groups`) and
stage 07 (`run_pathways`) on the shared specimen curves (`clustering_first_inputs`), with:
- `CompareParams(prior_k = 0)`, `PathwayParams(min_genes = 10, max_genes = 300)` and node spacing 1/29;
- each specimen curve floored at half a count over the specimen's summed library, now without the mt reads;
- the zone cuts from the package's marker rule (Slc5a12, Slc13a3, Slc16a9) on the control mice of the condition
  split. They are computed on the unchanged log-normalized matrix. **Guard:** they equal notebook 66's.

**Runs.** The condition split with the default null (both relabelings; its calls) and with each single-split null;
each relabeling with the fair null (the other relabeling). The relabelings with the default null call nothing by
construction, so they are not run.

**One call** is a pathway with matched q < 0.05 in at least one gene group or cluster.

In [ ]:
import pseudospace_reconstruction as prc
from pseudospace_reconstruction.compare import CompareParams, Design, compare_groups
from pseudospace_reconstruction.compare.pathways import PathwayParams, ora_by_group, run_pathways
from pseudospace_reconstruction.features.axis import zone_cuts_from_markers

from pseudospace.pathway_pipelines import FLOOR_COUNTS, clustering_first_inputs, single_split_design

COMPARE_PARAMS = CompareParams(prior_k=0.)                                       # no prior in the log ratio
PATHWAY_PARAMS = PathwayParams(min_genes=SET_SIZES[0], max_genes=SET_SIZES[1])    # the package's cap is 150
package = Path(prc.__file__).parent
external_key = digest([code_key, *sorted((package / 'compare').glob('*.py')), *sorted((package / 'features').glob('*.py')),
                       package / 'io' / 'gene_sets.py'])
ZONES66 = pd.read_csv(nb66 / 'tables' / 'method_b_zones.csv', index_col='dataset')
ZONES = {}
for ds, d in DATA.items():
    reference = np.flatnonzero(~d['case'])
    found = zone_cuts_from_markers(d['lognorm'][reference], d['genes'], d['position'][reference], ZONE_MARKERS)
    ZONES[ds] = np.asarray(found['cuts'], float)
    np.testing.assert_allclose(ZONES[ds], ZONES66.loc[d['title'], ['S1|S2 cut', 'S2|S3 cut']].to_numpy(float), rtol=0, atol=1e-12)
guard_log.append("zone cuts = notebook 66's, both datasets")

EXPRESSION_COLUMN = 'expression_per_10k'
GENE_COLUMNS = ['gene', 'selected', 'shape_replicated', 'direction', 'shape_class', 'location', 'group', 'cluster', 'rms',
                'level', 'shape_rms', EXPRESSION_COLUMN]


def design_of(ds, label):
    d = DATA[ds]
    per = pd.Series(PARTITIONS[ds][label][0], index=d['specimen']).groupby(level=0).first()
    return Design.from_labels(list(per.index), list(np.where(per.eq(1), 'case', 'reference')), case_label='case',
                              reference_label='reference')


def run_b(ds, contrast, null):
    """The package's stage 06 (compare_groups) and stage 07 (run_pathways) on the shared curves (notebook 66)."""
    d = DATA[ds]
    axis, fit = clustering_first_inputs(MODELS[ds][contrast], d['position'], PARTITIONS[ds][contrast][0], d['specimen'],
                                        d['library'], zone_names=list(ZONE_MARKERS), zone_cuts=ZONES[ds],
                                        zone_short_names=ZONE_SHORT)
    design = design_of(ds, contrast)
    if null is not None:
        design = single_split_design(design, group_one(ds, null))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)     # the package's own all-NaN / inf guards
        result = compare_groups(fit, axis, design, node_spacing=NODE_SPACING, elsewhere_flags=None, params=COMPARE_PARAMS)
        gt = result.gene_table
        curves = result.curves()                            # float32, as its stage 07 reads them from curves.npz
        pathways = run_pathways(curves['D'].astype(float), curves['P'].astype(float), curves['N'].astype(float),
                                curves['weights'], curves['grid'], EXT_SETS[ds], gt[EXPRESSION_COLUMN].to_numpy(),
                                gt['group'].to_numpy(), cluster_labels=gt['cluster'].to_numpy(), flagged=None,
                                groups=list(result.shape_groups.index), pair_names=[str(n) for n in curves['pair_names']],
                                split_names=[str(n) for n in curves['null_names']], pair_samples=design.pairs,
                                params=PATHWAY_PARAMS)
    genes = gt[GENE_COLUMNS].copy()
    for c in ('selected', 'shape_replicated'):
        genes[c] = genes[c].fillna(False).astype(bool)
    for c in ('direction', 'shape_class', 'location', 'group', 'cluster'):
        genes[c] = genes[c].fillna('').astype(str)
    ora = pd.concat([t.assign(list=kind) for kind, t in (('group', pathways.by_group), ('cluster', pathways.by_cluster))],
                    ignore_index=True)
    ora = ora[['list', 'group', 'n_genes', 'library', 'pathway', 'set_size', 'overlap', 'fold_matched', 'p_plain', 'p_matched',
               'q_plain', 'q_matched', 'overlap_genes']].astype({'group': str, 'library': str, 'pathway': str, 'overlap_genes': str})
    ora.insert(0, 'pathway_id', ora.library + '::' + ora.pathway)
    calls = pathways.calls
    calls = pd.DataFrame({'pathway_id': (calls.library + '::' + calls.pathway).to_numpy(), 'pathway_call': calls.pathway_call.astype(str),
                          'is_representative': calls.is_representative.astype(bool), 'level_es': calls.level_es.astype(float),
                          'disruption_z_loo': calls.disruption_z_loo.astype(float), 'coherence': calls.coherence.astype(float)})
    info = pd.DataFrame({'null_names': [' | '.join(design.null_names)], 'n_splits': [len(design.mixed_splits)]})
    return {'genes': genes, 'ora': ora, 'calls': calls, 'info': info}


def flatten(tables_):
    out = {}
    for name, frame in tables_.items():
        out[f'{name}|__columns'] = np.array(list(frame.columns), dtype=str)
        for column in frame.columns:
            values = frame[column].to_numpy()
            out[f'{name}|{column}'] = values.astype(str) if values.dtype == object else values
    return out


def unflatten(payload):
    names = sorted({k.split('|', 1)[0] for k in payload if not k.startswith('__')})   # '__seconds': the stage's run time
    return {n: pd.DataFrame({c: payload[f'{n}|{c}'] for c in payload[f'{n}|__columns'].astype(str)}) for n in names}


B, B_STAGE = {ds: {} for ds in DATASETS}, {ds: {} for ds in DATASETS}
for ds, d in DATA.items():
    condition, r1, r2 = list(PARTITIONS[ds])
    curves_key = digest({n: c for n, c in MODELS[ds]['species'].specimen_curves.items()})
    for contrast, null in [(condition, None), (condition, r1), (condition, r2), (r1, r2), (r2, r1)]:
        stage = f'method_b_{ds}_{slug(contrast)}__null_{"default" if null is None else slug(null)}'
        payload = cached(stage, lambda ds=ds, contrast=contrast, null=null: flatten(run_b(ds, contrast, null)),
                         params={'compare': repr(COMPARE_PARAMS), 'pathways': repr(PATHWAY_PARAMS),
                                 'node_spacing': NODE_SPACING, 'floor': FLOOR_COUNTS},
                         inputs={**d['input_key'], 'curves': curves_key, 'group': PARTITIONS[ds][contrast][0],
                                 'zones': ZONES[ds], 'sets': d['sets'], 'null': 'default' if null is None else group_one(ds, null)},
                         key=external_key)
        B[ds][(contrast, null)] = unflatten(payload)
        B_STAGE[ds][(contrast, null)] = stage


def b_calls(t):
    """The clustering method's calls of one run: matched ORA q < 0.05 in a gene group or a cluster."""
    sig = t['ora'][t['ora'].q_matched.astype(float) < ALPHA]
    return set(sig.pathway_id)


def b_bonferroni_p(ds, t):
    """Notebook 61/66's list-based convention: the best matched ORA p over the lists x the number of lists."""
    genes = t['genes']
    n_lists = sum(int((genes[c][genes[c].ne('')].value_counts() >= PATHWAY_PARAMS.min_group_size).sum()) for c in ('group', 'cluster'))
    best = t['ora'].groupby('pathway_id').p_matched.min().astype(float)
    return (best * max(n_lists, 1)).clip(upper=1.).reindex(list(DATA[ds]['sets'])).fillna(1.).to_numpy()


RUN_ROWS = []
for ds in DATASETS:
    for (contrast, null), t in B[ds].items():
        genes = t['genes']
        selected = genes[genes.selected.astype(bool)]
        RUN_ROWS.append({'dataset': DATA[ds]['title'], 'contrast': SHORT[ds][contrast],
                         'null': 'default (both other partitions)' if null is None else SHORT[ds][null],
                         'selected genes': len(selected), 'shape confirmed': int(genes.shape_replicated.astype(bool).sum()),
                         'up / down / crossover': '{} / {} / {}'.format(*(int(selected.direction.eq(x).sum()) for x in ('up', 'down', 'crossover'))),
                         'ORA calls': len(b_calls(t)), 'redundancy clusters': len(representatives(ds, b_calls(t))),
                         'seconds': round(SECONDS[B_STAGE[ds][(contrast, null)]], 1)})
RUN_TABLE = pd.DataFrame(RUN_ROWS)
RUN_TABLE.to_csv(tables / 'clustering_runs.csv', index=False)
display(RUN_TABLE)

## 5 · Check (c): calls and pass rules

The columns are notebook 66's:
- **calls:** pathways called for the condition split; the next column gives the redundancy clusters they touch;
- **relabeled calls:** calls under the two relabelings (the fake-group calls; clustering: with the fair null);
- **NCDR:** mean relabeled calls ÷ condition calls;
- **relabeling rule** (notebook 31): NCDR < 0.25 and each relabeling below 5% of the tested pathways;
- **calibration share:** of the 2 × (tested pathways) pooled relabeled one-sided p-values, the share ≤ 0.05; the limit
  is 0.075. The clustering method has no p-value per pathway; its line uses the list-based convention.

The rows marked **66 ·** and **67 ·** are read from those notebooks' method tables: the "before".

**Expected (protocol):** the normalized method still passes on human vs mouse and still fails calibration on AKI;
the clustering method stays specific under the fair null, with its calls nearly unchanged.

In [ ]:
METHOD_ROWS = []
N_LABEL = 'normalized · cameraPR on z'
B_LABEL, B_SINGLE_LABEL = 'clustering · ORA, default null', 'clustering · ORA, single-split null'
for ds in DATASETS:
    d = DATA[ds]
    condition, r1, r2 = list(PARTITIONS[ds])
    tested = len(d['sets'])

    def add(label, condition_sets, relabeled_sets, p_relabeled, note=''):
        n_condition = [len(c) for c in condition_sets]
        n_relabeled = [len(r) for r in relabeled_sets]
        mean_condition = float(np.mean(n_condition))
        ncdr = np.mean(n_relabeled) / mean_condition if mean_condition else np.nan
        share = float(np.mean(np.asarray(p_relabeled, float) <= ALPHA))
        METHOD_ROWS.append({
            'dataset': d['title'], 'method': label, 'calls': ', '.join(map(str, n_condition)),
            'calls (redundancy clusters)': ', '.join(str(len(representatives(ds, c))) for c in condition_sets),
            'relabeled calls': ', '.join(map(str, n_relabeled)), 'NCDR': ncdr,
            'relabeling rule': 'pass' if mean_condition and ncdr < NCDR_MAX and max(n_relabeled) < NULL_RATE * tested else 'fail',
            'calibration share': share, 'calibration': ('pass' if share <= CALIBRATION_MAX else 'fail') + note})

    add(N_LABEL, [N_CALLS[ds][condition]], [N_CALLS[ds][r1], N_CALLS[ds][r2]],
        np.concatenate([TESTS_N[ds][r].p_one_sided.to_numpy(float) for r in (r1, r2)]))
    fair = [B[ds][(r1, r2)], B[ds][(r2, r1)]]
    p_fair = np.concatenate([b_bonferroni_p(ds, t) for t in fair])
    add(B_LABEL, [b_calls(B[ds][(condition, None)])], [b_calls(t) for t in fair], p_fair, ' (list-based convention)')
    add(B_SINGLE_LABEL, [b_calls(B[ds][(condition, r)]) for r in (r1, r2)], [b_calls(t) for t in fair], p_fair,
        ' (list-based convention)')
TABLE66, TABLE67 = pd.read_csv(nb66 / 'tables' / 'method_table.csv'), pd.read_csv(nb67 / 'tables' / 'method_table.csv')
BEFORE_ROWS = {'67 · ' + N_LABEL: (TABLE67, 'N · cameraPR on z (primary)'),
               '66 · ' + B_LABEL: (TABLE66, 'B · ORA on gene groups and clusters (his default null)'),
               '66 · ' + B_SINGLE_LABEL: (TABLE66, 'B · ORA, single-split null (like for like)')}
for ds in DATASETS:
    for label, (table, original) in BEFORE_ROWS.items():
        row = table[table.dataset.eq(DATA[ds]['title']) & table.method.eq(original)]
        assert len(row) == 1, (ds, original)
        METHOD_ROWS.append({**{k: row.iloc[0][k] for k in ('calls', 'calls (redundancy clusters)', 'relabeled calls', 'NCDR',
                                                            'relabeling rule', 'calibration share', 'calibration')},
                            'dataset': DATA[ds]['title'], 'method': label})
METHOD_TABLE = pd.DataFrame(METHOD_ROWS)[['dataset', 'method', 'calls', 'calls (redundancy clusters)', 'relabeled calls', 'NCDR',
                                          'relabeling rule', 'calibration share', 'calibration']]
METHOD_TABLE.to_csv(tables / 'method_table.csv', index=False)
PASS = {}
for ds in DATASETS:
    rows = METHOD_TABLE[METHOD_TABLE.dataset.eq(DATA[ds]['title'])].set_index('method')
    PASS[ds] = rows.loc[N_LABEL, 'relabeling rule'] == 'pass' and rows.loc[N_LABEL, 'calibration'].startswith('pass')
    print(f"{DATA[ds]['title']}: {len(DATA[ds]['sets']):,} pathways tested; the normalized method "
          f"{'passes' if PASS[ds] else 'fails'} the pass rule")
    display(rows.drop(columns='dataset').round(3))

## 6 · Checks (a) and (b): null drift and direction balance

**d_g** is the mean of Δ_g(x) over the last 21 of the 101 grid points (the last 20% of the grid's range) minus the
mean over the first 21. Δ is the natural-log rate ratio, case over reference, from the condition split; d_g > 0
means "rising in human (AKI) toward S3". Only converged, non-separated genes enter (the gene model's flags).
- **Before:** notebook 67's cached condition-split gene model (Δ and flags; it equals notebook 66's) and notebook 67's
  gene-level q.
- **After:** this notebook.

**(a) Null drift.** The median of d_g over the genes with q > 0.5 on the normalized method in both runs (the mt genes
are absent after, so excluded from both). **Criterion** (human vs mouse): |after| ≤ 0.05. AKI is reported, not judged.

Beside it:
- **the predicted "after":** the "before" plus the tilt the mt shares imply. Within the first and the last 20% of
  the grid's range, each specimen's mt share m_j is its summed mt counts over its summed old library. Removing them
  shifts that specimen's log rates by −log(1 − m_j), so the window's Δ shifts by the case specimens' mean of
  −log(1 − m_j) minus the reference specimens' mean. The tilt is the last window's shift minus the first window's;
- the same median over each run's own q > 0.5 genes.

**(b) Direction balance.** Among genes with q ≤ 0.05 (condition split) in each run, the number with d_g > 0 and with
d_g < 0. **Expected:** the rising share falls on human vs mouse. Descriptive.

In [ ]:
K_WINDOW = int(round(DRIFT_SHARE * (GRID_POINTS - 1))) + 1      # 21 of the 101 grid points at each end


def drift(delta):
    """d_g: mean delta over the last 20% of the grid minus the mean over the first 20% (genes,)."""
    delta = np.asarray(delta, float)
    return delta[:, -K_WINDOW:].mean(axis=1) - delta[:, :K_WINDOW].mean(axis=1)


def mt_tilt(ds):
    """The change in d_g that removing the mt reads implies from the library shares alone (see the text above)."""
    d, grid = DATA[ds], MODELS[ds]['species'].grid
    windows = {'first 20%': (grid[0], grid[K_WINDOW - 1]), 'last 20%': (grid[-K_WINDOW], grid[-1])}
    mt = d['library_with_mt'] - d['library']
    rows, shift = [], {}
    for window, (lo, hi) in windows.items():
        inside = (d['position'] >= lo) & (d['position'] <= hi)
        share = {s: mt[inside & (d['specimen'] == s)].sum() / d['library_with_mt'][inside & (d['specimen'] == s)].sum()
                 for s in d['case_specimens'] + d['reference_specimens']}
        shift[window] = (np.mean([-np.log1p(-share[s]) for s in d['case_specimens']])
                         - np.mean([-np.log1p(-share[s]) for s in d['reference_specimens']]))
        rows += [{'dataset': d['title'], 'window': window, 'specimen': s,
                  'group': d['case_name'] if s in d['case_specimens'] else d['reference_name'],
                  'structures': int((inside & (d['specimen'] == s)).sum()), 'mt share of the old library': share[s]}
                 for s in share]
    return shift['last 20%'] - shift['first 20%'], rows


assert abs((MODELS[DATASETS[0]]['species'].grid[K_WINDOW - 1] - MODELS[DATASETS[0]]['species'].grid[0])
           / np.ptp(MODELS[DATASETS[0]]['species'].grid) - DRIFT_SHARE) < 1e-9
D_BEFORE, D_AFTER, OK_BEFORE, OK_AFTER, Q_BEFORE, Q_AFTER = {}, {}, {}, {}, {}, {}
DRIFT_ROWS, BALANCE_ROWS, SHARE_ROWS, TILT = [], [], [], {}
for ds, d in DATA.items():
    before, model = BEFORE_MODEL[ds], MODELS[ds]['species']
    q67 = pd.read_csv(nb67 / 'tables' / f'gene_statistics_{ds}_primary.csv', header=[0, 1], index_col=0)[('condition', 'q')]
    D_BEFORE[ds] = pd.Series(drift(before['delta']), index=KEPT[ds]['genes'])
    OK_BEFORE[ds] = pd.Series(before['stat|converged'].astype(bool) & ~before['stat|separated'].astype(bool), index=KEPT[ds]['genes'])
    Q_BEFORE[ds] = q67.astype(float).reindex(KEPT[ds]['genes'])
    D_AFTER[ds] = pd.Series(drift(model.delta), index=d['genes'])
    OK_AFTER[ds] = model.statistics.converged & ~model.statistics.separated
    Q_AFTER[ds] = STAT[ds]['species'].q
    TILT[ds], rows = mt_tilt(ds)
    SHARE_ROWS += rows
    genes = pd.Index(d['genes'])                           # non-mt genes, present in both runs
    ok = OK_BEFORE[ds].reindex(genes) & OK_AFTER[ds]
    null_both = ok & Q_BEFORE[ds].reindex(genes).gt(NULL_Q) & Q_AFTER[ds].gt(NULL_Q)
    null_before = OK_BEFORE[ds] & Q_BEFORE[ds].gt(NULL_Q) & ~OK_BEFORE[ds].index.isin(MT)
    null_after = OK_AFTER[ds] & Q_AFTER[ds].gt(NULL_Q)
    d['null_genes'] = list(genes[null_both.to_numpy()])
    b, a = float(D_BEFORE[ds][d['null_genes']].median()), float(D_AFTER[ds][d['null_genes']].median())
    judged = ds == 'human_vs_mouse'
    DRIFT_ROWS.append({'dataset': d['title'], 'genes (q > 0.5 in both runs)': len(d['null_genes']),
                       'median d_g before': b, 'median d_g after': a, 'predicted after (before + mt tilt)': b + TILT[ds],
                       'mt tilt': TILT[ds], 'criterion |after| ≤ 0.05': ('pass' if abs(a) <= DRIFT_LIMIT else 'fail') if judged else 'reported only',
                       'own q > 0.5 genes: before (n)': f"{D_BEFORE[ds][null_before].median():.4f} ({int(null_before.sum()):,})",
                       'own q > 0.5 genes: after (n)': f"{D_AFTER[ds][null_after].median():.4f} ({int(null_after.sum()):,})"})
    for run, dd, okk, qq in (('before (notebook 67)', D_BEFORE[ds], OK_BEFORE[ds], Q_BEFORE[ds]),
                             ('after (notebook 69)', D_AFTER[ds], OK_AFTER[ds], Q_AFTER[ds])):
        sig = dd[okk & qq.le(GENE_FDR)]
        BALANCE_ROWS.append({'dataset': d['title'], 'run': run, 'significant genes (q ≤ 0.05)': len(sig),
                             f'rising in case toward S3 (d_g > 0)': int((sig > 0).sum()), 'falling (d_g < 0)': int((sig < 0).sum()),
                             'share rising': float((sig > 0).mean()), 'median d_g': float(sig.median()),
                             'of which mt genes': int(sig.index.isin(MT).sum())})
DRIFT = pd.DataFrame(DRIFT_ROWS)
BALANCE = pd.DataFrame(BALANCE_ROWS)
SHARES = pd.DataFrame(SHARE_ROWS)
DRIFT.to_csv(tables / 'check_a_null_drift.csv', index=False)
BALANCE.to_csv(tables / 'check_b_direction_balance.csv', index=False)
SHARES.to_csv(tables / 'mt_share_windows.csv', index=False)
display(SHARES.pivot_table(index=['dataset', 'group', 'specimen'], columns='window', values='mt share of the old library', sort=False).round(4))
display(DRIFT.set_index('dataset').round(4))
display(BALANCE.set_index(['dataset', 'run']).round(3))
CHECK_A = DRIFT.set_index('dataset').loc[DATA['human_vs_mouse']['title'], 'criterion |after| ≤ 0.05']
print(f"Check (a), human vs mouse: {CHECK_A.upper()} (|after| = {abs(DRIFT.iloc[0]['median d_g after']):.4f}, limit {DRIFT_LIMIT})")
pd.DataFrame({'d_g before': pd.concat(D_BEFORE), 'd_g after': pd.concat(D_AFTER)}).rename_axis(['dataset', 'gene']).to_csv(
    tables / 'gene_drift_before_after.csv')

## 7 · Check (d): overlap between the methods

The normalized calls (cameraPR on z) against the clustering calls (its ORA with the default null, notebook 66's
"B ORA"), for the condition split:
- **both**, **normalized only**, **clustering only** and **neither** (tested pathways called by neither); the same
  over redundancy clusters, where "neither" counts the clusters that no call of either method touches;
- the Jaccard index over pathways and over redundancy clusters.

**Without the 8 proteasome subunits** (Psma3, Psma4, Psmb5, Psmb6, Psmc2, Psmc6, Psmd11, Psmd14), as in the earlier
sensitivity:
- **normalized:** cameraPR on the same z without the 8 genes, with no squeezeVar refit; sets below 10 members are
  dropped;
- **clustering:** its ORA (`ora_by_group` with its `PathwayParams`) rerun with the 8 genes outside the universe, on
  the gene groups and clusters of the condition run, unchanged. **Guard:** with the full universe it reproduces the
  run's own calls.

The "before" rows (notebooks 66 and 67) are recomputed the same way from their saved z, gene groups and calls.
**Expected:** the Jaccard over redundancy clusters stays ≤ 0.2 on both datasets, with and without the 8 subunits.

In [ ]:
ORA_KW = dict(n_strata=PATHWAY_PARAMS.n_strata, min_set_size=PATHWAY_PARAMS.min_genes,
              min_group_size=PATHWAY_PARAMS.min_group_size, min_overlap=PATHWAY_PARAMS.min_overlap)


def clustering_calls(genes_table, sets, universe=None):
    """The clustering method's ORA calls from a run's gene groups and clusters (the package's ora_by_group)."""
    expression = genes_table[EXPRESSION_COLUMN].to_numpy(float)
    ora = pd.concat([ora_by_group(genes_table[k].to_numpy(), sets, expression, universe=universe, **ORA_KW).assign(list=k)
                     for k in ('group', 'cluster')], ignore_index=True)
    return set((ora.library + '::' + ora.pathway)[ora.q_matched < ALPHA])


def without_proteasome(sets):
    out = {p: [g for g in v if g not in PROTEASOME] for p, v in sets.items()}
    return {p: v for p, v in out.items() if len(v) >= SET_SIZES[0]}


CALLSETS = {}
for ds, d in DATA.items():
    condition = list(PARTITIONS[ds])[0]
    psm = [g for g in PROTEASOME if g in d['genes']]
    # After (this notebook)
    after_genes = B[ds][(condition, None)]['genes']
    assert list(after_genes.gene.astype(str)) == d['genes']
    assert clustering_calls(after_genes, EXT_SETS[ds]) == b_calls(B[ds][(condition, None)]), 'ORA does not reproduce the run'
    z_after = STAT[ds]['species'].z
    camera = camera_pr(z_after.drop(psm), without_proteasome(d['sets']), inter_gene_cor=CAMERA_COR, fdr=ALPHA)
    CALLSETS[(ds, 'notebook 69', 'all genes')] = (N_CALLS[ds][condition], b_calls(B[ds][(condition, None)]))
    CALLSETS[(ds, 'notebook 69', 'without the 8 proteasome subunits')] = (
        set(camera.pathway_id[camera.call]), clustering_calls(after_genes, EXT_SETS[ds], universe=~np.isin(d['genes'], psm)))
    # Before (notebooks 66 and 67), from their saved z, gene groups and calls
    kept_genes = KEPT[ds]['genes']
    before_genes = unflatten(stored(nb66, f'method_b_{ds}_species__null_default'))['genes']
    assert list(before_genes.gene.astype(str)) == kept_genes
    kept_sets = build_gene_sets({lib: {p.split('::', 1)[1]: v for p, v in KEPT[ds]['sets'].items() if p.startswith(lib + '::')}
                                 for lib in LIBRARIES}, kept_genes, min_genes=SET_SIZES[0], max_genes=SET_SIZES[1])
    detail66 = pd.read_csv(nb66 / 'tables' / f'pathway_detail_{ds}.csv', index_col='pathway_id')
    b_before = set(detail66.index[detail66['called: B ORA'].astype(bool)])
    assert clustering_calls(before_genes, kept_sets) == b_before, "ORA does not reproduce notebook 66's calls"
    test67 = pd.read_csv(nb67 / 'tables' / f'pathway_tests_{ds}_cameraPR_{slug(condition)}.csv')
    n_before = set(test67.pathway_id[test67.call.astype(bool)])
    z_before = pd.read_csv(nb67 / 'tables' / f'gene_statistics_{ds}_primary.csv', header=[0, 1], index_col=0)[('condition', 'z')].astype(float)
    camera = camera_pr(z_before.drop(psm), without_proteasome(KEPT[ds]['sets']), inter_gene_cor=CAMERA_COR, fdr=ALPHA)
    CALLSETS[(ds, 'notebooks 66/67', 'all genes')] = (n_before, b_before)
    CALLSETS[(ds, 'notebooks 66/67', 'without the 8 proteasome subunits')] = (
        set(camera.pathway_id[camera.call]), clustering_calls(before_genes, kept_sets, universe=~np.isin(kept_genes, psm)))
    guard_log.append(f"{d['title']}: the clustering ORA recomputed from the gene groups reproduces this run's and notebook 66's "
                     f"calls; {len(psm)} of the 8 proteasome subunits are tested genes")

OVERLAP_ROWS = []
for (ds, run, universe), (n_calls, b_calls_) in CALLSETS.items():
    all_clusters = set(REDUNDANCY[ds].representative)
    for unit, x, y, total in (('pathways', n_calls, b_calls_, set(DATA[ds]['sets'])),
                              ('redundancy clusters', representatives(ds, n_calls), representatives(ds, b_calls_), all_clusters)):
        OVERLAP_ROWS.append({'dataset': DATA[ds]['title'], 'run': run, 'universe': universe, 'unit': unit,
                             'normalized calls': len(x), 'clustering calls': len(y), 'both': len(x & y),
                             'normalized only': len(x - y), 'clustering only': len(y - x), 'neither': len(total - (x | y)),
                             'Jaccard': len(x & y) / len(x | y) if x | y else np.nan})
OVERLAP = pd.DataFrame(OVERLAP_ROWS)
OVERLAP.to_csv(tables / 'check_d_overlap.csv', index=False)
display(OVERLAP.set_index(['dataset', 'universe', 'unit', 'run']).sort_index(level=[0, 1, 2], sort_remaining=False).round(3))
print('\n'.join('· ' + line for line in guard_log[-2:]))

## 8 · Check (e): lost and gained calls

For each method and dataset, the calls of this notebook against notebook 67's (normalized) and notebook 66's
default-null ORA (clustering). For each lost or gained pathway:
- its FDR (normalized) or best matched q over the lists (clustering), before and after;
- the median d_g over its tested members, before and after: its median direction toward S3 (> 0: rising in the case
  group).

**Expected:** the normalized method's lost calls are mostly rising pathways (median d_g before > 0) and its gained
calls mostly falling ones; the clustering method loses and gains few calls.

In [ ]:
def member_drift(ds, values, ok, pathway):
    members = [g for g in DATA[ds]['sets'][pathway] if g in values.index and ok.get(g, False)]
    return float(values[members].median()) if members else np.nan


def best_q(t):
    return t['ora'].astype({'q_matched': float}).groupby('pathway_id').q_matched.min()


CHANGE_ROWS, CHANGE_SUMMARY = [], []
for ds, d in DATA.items():
    condition = list(PARTITIONS[ds])[0]
    detail66 = pd.read_csv(nb66 / 'tables' / f'pathway_detail_{ds}.csv', index_col='pathway_id')
    test67 = pd.read_csv(nb67 / 'tables' / f'pathway_tests_{ds}_cameraPR_{slug(condition)}.csv').set_index('pathway_id')
    test69 = TESTS_N[ds][condition].set_index('pathway_id')
    q66, q69 = detail66['B best q'], best_q(B[ds][(condition, None)])
    sides = {'normalized': (CALLSETS[(ds, 'notebooks 66/67', 'all genes')][0], N_CALLS[ds][condition], test67.fdr, test69.fdr, 'FDR'),
             'clustering': (CALLSETS[(ds, 'notebooks 66/67', 'all genes')][1], b_calls(B[ds][(condition, None)]), q66, q69, 'best q')}
    for method, (before, after, s_before, s_after, stat_name) in sides.items():
        for change, members in (('lost', before - after), ('gained', after - before)):
            for p in sorted(members):
                CHANGE_ROWS.append({'dataset': d['title'], 'method': method, 'change': change, 'pathway_id': p, 'pathway': NAMES[ds][p],
                                    'genes': len(d['sets'][p]), f'{stat_name} before': s_before.get(p, np.nan),
                                    f'{stat_name} after': s_after.get(p, np.nan),
                                    'median d_g before': member_drift(ds, D_BEFORE[ds], OK_BEFORE[ds], p),
                                    'median d_g after': member_drift(ds, D_AFTER[ds], OK_AFTER[ds], p)})
        rows = [r for r in CHANGE_ROWS if r['dataset'] == d['title'] and r['method'] == method]
        lost = [r for r in rows if r['change'] == 'lost']
        gained = [r for r in rows if r['change'] == 'gained']
        CHANGE_SUMMARY.append({'dataset': d['title'], 'method': method, 'calls before': len(before), 'calls after': len(after),
                               'kept': len(before & after), 'lost': len(lost), 'gained': len(gained),
                               'lost, rising before (median d_g > 0)': sum(r['median d_g before'] > 0 for r in lost),
                               'gained, falling after (median d_g < 0)': sum(r['median d_g after'] < 0 for r in gained),
                               'Jaccard before/after (pathways)': len(before & after) / len(before | after),
                               'Jaccard before/after (redundancy clusters)':
                                   len(representatives(ds, before) & representatives(ds, after))
                                   / len(representatives(ds, before) | representatives(ds, after))})
CHANGES = pd.DataFrame(CHANGE_ROWS)
SUMMARY_E = pd.DataFrame(CHANGE_SUMMARY)
for (ds_title, method), part in CHANGES.groupby(['dataset', 'method'], sort=False):
    ds = next(k for k in DATASETS if DATA[k]['title'] == ds_title)
    part.dropna(axis=1, how='all').to_csv(tables / f'check_e_lost_gained_{method}_{ds}.csv', index=False)
SUMMARY_E.to_csv(tables / 'check_e_summary.csv', index=False)
display(SUMMARY_E.set_index(['dataset', 'method']).round(3))
for (ds_title, method), part in CHANGES.groupby(['dataset', 'method'], sort=False):
    print(f'\n{ds_title} · {method}: lost {int(part.change.eq("lost").sum())}, gained {int(part.change.eq("gained").sum())}')
    with pd.option_context('display.max_colwidth', 60, 'display.max_rows', 60):
        display(part.drop(columns=['dataset', 'method', 'pathway_id']).dropna(axis=1, how='all')
                .sort_values(['change', 'median d_g before'], ascending=[False, False]).head(40).set_index('pathway').round(3))

### 8.1 · Post hoc: where the clustering calls changed

This subsection was added after check (e) showed that the clustering method lost 36 human vs mouse calls, although
its typical-gene re-referencing should absorb a shift shared by all genes. It changes no call and no verdict above.

For each condition run of the clustering method, before (notebook 66's cache) and after, the table gives:
- the calls, and the list (gene group or shape cluster) that carries most of them, with its genes and calls;
- the shape clusters that hold the 8 proteasome subunits;
- the calls through the gene group "Down · gradient toward S3", which does not depend on the Ward clustering.

In [ ]:
LIST_ROWS = []
for ds in DATASETS:
    condition, r1, r2 = list(PARTITIONS[ds])
    for null in (None, r1, r2):
        stage = f'method_b_{ds}_{slug(condition)}__null_{"default" if null is None else slug(null)}'
        for run, t in (('notebook 66', unflatten(stored(nb66, stage))), ('notebook 69', B[ds][(condition, null)])):
            genes, ora = t['genes'], t['ora']
            sig = ora[ora.q_matched.astype(float) < ALPHA]
            per = sig.groupby(['list', 'group']).pathway_id.nunique().sort_values(ascending=False)
            (kind, name), top = per.index[0], int(per.iloc[0])
            holders = genes.set_index('gene').reindex(PROTEASOME).cluster.replace('', 'not selected').value_counts()
            LIST_ROWS.append({'dataset': DATA[ds]['title'], 'null': 'default' if null is None else SHORT[ds][null], 'run': run,
                              'calls': sig.pathway_id.nunique(),
                              'list carrying most calls': f'{kind}: {name} ({int(genes[kind].eq(name).sum())} genes)',
                              'its calls': top, 'clusters holding the 8 subunits': ', '.join(f'{c} ({n})' for c, n in holders.items()),
                              'calls through "Down · gradient toward S3"': int(per.get(('group', 'Down · gradient toward S3'), 0))})
LISTS = pd.DataFrame(LIST_ROWS)
LISTS.to_csv(tables / 'posthoc_clustering_lists.csv', index=False)
with pd.option_context('display.max_colwidth', 50):
    display(LISTS.set_index(['dataset', 'null', 'run']))

## 9 · The figure

**Panels a and b: the null drift.** For the genes of check (a) (q > 0.5 in both runs), the median over genes of
Δ_g(x) minus that gene's mean over the first 20% of the grid, along the PT. The curve starts near 0; its value over
the last 20% is close to the median d_g. Dashed: before (notebook 67); solid: after (this notebook). The shaded
bands are the two 20% windows; the dotted lines mark ±0.05, the check's limit at the S3 end.

**Panels c and d: overlap between the methods** (condition split, all genes), as a 2 × 2 count of pathways: rows
say whether the normalized method calls the pathway, columns whether the clustering method does. Each cell gives the
count after the change and, in brackets, before (notebooks 66/67).

In [ ]:
fig = plt.figure(figsize=(180 * MM, 124 * MM))
layout = fig.add_gridspec(2, 2, height_ratios=[1.1, 1], hspace=.62, wspace=.4)
for k, (ds, tag) in enumerate(zip(DATASETS, 'ab')):
    ax = fig.add_subplot(layout[0, k])
    d, grid = DATA[ds], MODELS[ds]['species'].grid
    genes = d['null_genes']
    for run, delta, index, color, ls in (('before', BEFORE_MODEL[ds]['delta'], KEPT[ds]['genes'], BEFORE_COLOR, '--'),
                                         ('after', MODELS[ds]['species'].delta, d['genes'], AFTER_COLOR, '-')):
        rows = pd.Index(index).get_indexer(genes)
        values = delta[rows] - delta[rows, :K_WINDOW].mean(axis=1, keepdims=True)
        curve = np.median(values, axis=0)
        ax.plot(grid, curve, color=color, ls=ls, lw=1.2)
        value = DRIFT.set_index('dataset').loc[d['title'], f'median d_g {run}']
        ax.text(grid[-1] + .015 * np.ptp(grid), curve[-1], f'{run}: {value:+.3f}', color=INK, fontsize=5.4, va='center')
    for lo, hi in ((grid[0], grid[K_WINDOW - 1]), (grid[-K_WINDOW], grid[-1])):
        ax.axvspan(lo, hi, color=LIGHT, alpha=.45, lw=0, zorder=0)
    ax.axhline(0, color=LIGHT, lw=.6, zorder=0)
    for y in (-DRIFT_LIMIT, DRIFT_LIMIT):
        ax.plot([grid[-K_WINDOW], grid[-1]], [y, y], color=MUTED, lw=.6, ls=':')
    row = DRIFT.set_index('dataset').loc[d['title']]
    verdict = (f"criterion |after| ≤ {DRIFT_LIMIT}: {row['criterion |after| ≤ 0.05']}" if ds == 'human_vs_mouse'
               else 'reported, not judged')
    ax.text(.03, .03, f"predicted after: {row['predicted after (before + mt tilt)']:+.3f}\n{verdict}", transform=ax.transAxes,
            fontsize=5.4, color=MUTED, va='bottom')
    ax.set_title(f"{tag}  {d['title']}: null drift\n{len(genes):,} genes with q > 0.5 in both runs", loc='left')
    ax.set_xlabel('position along the PT (S1 → S3)')
    ax.set_ylabel(f"median Δ(x) − its S1-end mean\n(ln, {d['case_name']}/{d['reference_name']})")
    ax.set_xlim(grid[0], grid[-1] + .3 * np.ptp(grid))
    ax.set_xticks([grid[0], grid[-1]], [f'{grid[0]:.2f}', f'{grid[-1]:.2f}'])
OVERLAP_INDEXED = OVERLAP.set_index(['dataset', 'run', 'universe', 'unit'])
for k, (ds, tag) in enumerate(zip(DATASETS, 'cd')):
    ax = fig.add_subplot(layout[1, k])
    after = OVERLAP_INDEXED.loc[(DATA[ds]['title'], 'notebook 69', 'all genes', 'pathways')]
    before = OVERLAP_INDEXED.loc[(DATA[ds]['title'], 'notebooks 66/67', 'all genes', 'pathways')]
    cells = {(0, 0): 'both', (0, 1): 'normalized only', (1, 0): 'clustering only', (1, 1): 'neither'}
    for (r, c), key in cells.items():
        ax.add_patch(plt.Rectangle((c, 1 - r), 1, 1, facecolor='#e2ece6' if key == 'both' else '#f0efec', edgecolor='white', lw=2))
        ax.text(c + .5, 1.5 - r, f"{int(after[key]):,}\n({int(before[key]):,})", ha='center', va='center', fontsize=6.5, color=INK)
    ax.set_xlim(0, 2)
    ax.set_ylim(0, 2)
    ax.set_aspect('equal')
    ax.set_xticks([.5, 1.5], ['called', 'not called'])
    ax.set_yticks([1.5, .5], ['called', 'not called'])
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xlabel('clustering', color=B_COLOR, fontweight='bold')
    ax.set_ylabel('normalized', color=N_COLOR, fontweight='bold')
    ax.xaxis.set_label_position('top')
    ax.set_title(f"{tag}  {DATA[ds]['title']}: pathways called\nJaccard {after['Jaccard']:.2f} (before {before['Jaccard']:.2f})",
                 loc='left', pad=16)
fig.text(.5, .01, 'Panels c and d: count after the change (before: notebooks 66/67, mt reads in the library).',
         ha='center', fontsize=5.4, color=MUTED)
save(fig, 'fig1_null_drift_and_overlap')
plt.show()

## 10 · Readings of the protocol

Where the protocol left a choice, the most literal reading was taken. Each was fixed before any fit.
- **The mt genes** are the ortholog-map mouse symbols with the prefix `mt-` (13). They leave the library wherever
  they are measured: 12 in human vs mouse (mt-Atp8 is not measured in human) and 13 in AKI. They leave the universe
  where they were tested: 12 in each dataset.
- **The log-normalized matrix** keeps its full-library normalization. It feeds only the zone cuts, which are guarded
  to equal notebook 66's.
- **d_g's windows** are the first and last 21 of the 101 grid points (20% of the grid's range each).
- **"Before" flags** for checks (a), (b) and (e) are the gene model's converged and separated flags from notebook
  67's cache; the "after" flags are this notebook's gene model's. The gene-level q of the normalized method is
  computed on all genes as in notebook 67.
- **The predicted "after"** of check (a) uses the windows of d_g. Each specimen's mt share is pooled over its
  structures in the window (summed mt counts over the summed old library), and each group is the equal-specimen mean
  of −log(1 − m), because the group curves are equal-specimen means.
- **The proteasome sensitivity** follows the earlier one exactly; it is computed for both datasets and for both the
  "before" and the "after" call sets.
- **"Neither"** over redundancy clusters counts the clusters that no call of either method touches.
- **Not run:** notebook 66's cameraPR and GSEA on raw `T_spatial`, the joint test, the set-curve calls, the
  relabeled clustering runs with the default null, and notebook 67's sensitivities.

## 11 · Summary

**The change.** The mitochondrially encoded genes left the library: 12 in human vs mouse and 13 in AKI. They also
left each tested universe (12 genes). Every other input is notebook 66/67's: with the mt genes kept, the inputs
reproduce those notebooks' recorded stage keys. The pathways, redundancy clusters, zone cuts and grid are unchanged.

**Check (a) passes narrowly on human vs mouse.**
- Over the 1,875 genes with q > 0.5 in both runs, the median d_g is +0.079 before and −0.048 after. The limit is
  0.05. The mt shares predicted −0.055.
- The change removes the tilt and slightly overshoots it. Over each run's own q > 0.5 genes, the drift is +0.039
  before and +0.001 after.
- AKI (reported only): +0.016 before and −0.002 after.

**(b) Direction balance** (descriptive):
- Human vs mouse: before, 2,624 of the 3,835 significant genes rose in human toward S3 (68%), including all 12 mt
  genes. After, 1,890 of 3,614 (52%).
- AKI: 55% before and 51% after.

**(c) Calls and pass rules.** Each cell gives the calls (redundancy clusters); relabeled calls; NCDR; calibration
share. The "before" is notebooks 66/67.

| Method | Human vs mouse, before → after | AKI vs control, before → after |
|---|---|---|
| Normalized, cameraPR on z | 70 (68); 0, 0; 0; 0.044 → **97 (90); 0, 0; 0; 0.044 · passes** | 93 (86); 17, 11; 0.15; 0.102 → **99 (91); 16, 10; 0.13; 0.101 · fails calibration** |
| Clustering, default null | 167 (72); 0, 0 → **132 (51); 0, 0** | 464 (262); 2, 0 → **455 (253); 2, 0** |
| Clustering, single-split nulls | 162, 159 → 162, 46 | 469, 482 → 465, 468 |

- The normalized method still passes on human vs mouse and still fails calibration on AKI, as expected.
- The clustering method stays specific under the fair null.
- Its human vs mouse calls did **not** stay nearly unchanged, against the protocol's expectation. Section 8.1
  explains why.

**(d) Overlap between the methods** (condition split; pathways, then redundancy clusters). Each cell gives
both / normalized only / clustering only / neither, then the Jaccard index.

| Dataset | Genes | Before (66/67) | After (69) |
|---|---|---|---|
| Human vs mouse | all | 17 / 53 / 150 / 1,293 (J 0.077); clusters J 0.138 | 22 / 75 / 110 / 1,306 (J 0.106); clusters J 0.175 |
| Human vs mouse | without the 8 proteasome subunits | 14 / 57 / 21 / 1,421 (J 0.152); clusters J 0.161 | 20 / 77 / 14 / 1,402 (J 0.180); clusters J 0.198 |
| AKI vs control | all | 33 / 60 / 431 / 905 (J 0.063); clusters J 0.105 | 30 / 69 / 425 / 905 (J 0.057); clusters J 0.096 |
| AKI vs control | without the 8 proteasome subunits | 34 / 59 / 430 / 906 (J 0.065); clusters J 0.108 | 29 / 70 / 423 / 907 (J 0.056); clusters J 0.093 |

The overlap stays low, as expected: the Jaccard over redundancy clusters is ≤ 0.2 everywhere.

**(e) Lost and gained calls.**
- **Normalized, human vs mouse:** 69 kept, 1 lost, 28 gained.
  - The lost call is KRAS Signaling Up (FDR 0.047 → 0.052; rising in human).
  - 24 of the 28 gains had FDR 0.05–0.10 before.
  - 20 of the 28 fall in human toward S3 after the change. 14 already fell before.
  - Gains include cholesterol and bile acid synthesis, steroid metabolism, mTORC1 signalling, very-long-chain fatty
    acid β-oxidation, and glutathione synthesis.
  - The leading calls are unchanged: fatty acid metabolism, inorganic ion and amino acid transport, branched-chain
    amino acid degradation, xenobiotic and arginine/proline metabolism.
  - Spearman of pathway p before and after: 0.87.
- **Normalized, AKI:** 91 kept, 2 lost and 8 gained.
  - Lost: Myc Targets V1 and Processive Synthesis On Lagging Strand, both rising.
  - Gained, all falling: among them oxidative phosphorylation.
- **Clustering, human vs mouse:** 131 kept, 36 lost and 1 gained.
  - 26 of the 36 lost calls were rising before.
  - They are mostly Reactome sets that hold proteasome subunits (antigen processing, neddylation, deubiquitination,
    CLEC7A, the ER-phagosome pathway, IL-1 signalling), cell-cycle sets, p53 and EMT.
- **Clustering, AKI:** 443 kept, 21 lost and 12 gained. 24 of these 33 sit at a matched q of 0.05–0.09 on the other
  side.

**Post hoc (8.1): why the clustering calls moved.**
- **Before.** In every human vs mouse condition run, all 8 proteasome subunits sat in one Ward shape cluster, which
  carried 126–130 of the calls.
- **After.** The subunits are split between two clusters.
  - Default null: D4 with 126 calls → split 5 / 3.
  - Relabeling-2 null: D5 with 127 calls → split 4 / 4. That run falls from 159 to 46 calls.
  - Relabeling-1 null: the subunits split 6 / 2, the larger cluster still carries 129 calls, and the run keeps 162.
- **The stable part.** The gene group that does not depend on clustering, "Down · gradient toward S3", carries 107
  calls after and 108 before.
- **Without the 8 subunits** the clustering calls are 34 after and 35 before.
- **So the clustering count rests on whether the 8 subunits land in one cluster.** That is a discrete outcome of
  the Ward k rule, and the mt change moved it.

**Decision (protocol).** Check (a) passes, so these calls replace notebooks 66/67's as the primary for both methods
on both datasets. The normalized method's AKI calibration failure stands.

**What this cannot tell you.**
- Check (a) passes by 0.002. The mt tilt (−0.13) is larger than the drift it was meant to remove (+0.08), so the
  null genes now drift slightly the other way.
- Within each specimen the mt share still varies between structures, and the offset now ignores it.
- The relabelings mix the conditions, so they cannot see an artefact that follows the condition.
- With two specimens per group every result is descriptive. Section 8.1 was added after the results were seen.

In [ ]:
row = DRIFT.set_index('dataset').loc[DATA['human_vs_mouse']['title']]
print(f"Check (a), human vs mouse: median d_g of the null genes {row['median d_g before']:+.4f} before, "
      f"{row['median d_g after']:+.4f} after (predicted {row['predicted after (before + mt tilt)']:+.4f}); "
      f"criterion |after| ≤ {DRIFT_LIMIT}: {CHECK_A}.")
print('Decision rule: ' + ('notebook 69\'s calls replace notebooks 66/67\'s as the primary for both methods.' if CHECK_A == 'pass'
                           else 'check (a) fails; notebooks 66/67 stay primary and nothing is tuned.'))
for ds in DATASETS:
    rows = METHOD_TABLE[METHOD_TABLE.dataset.eq(DATA[ds]['title'])].set_index('method')
    for label in (N_LABEL, B_LABEL):
        r = rows.loc[label]
        print(f"{DATA[ds]['title']} · {label}: {r['calls']} calls ({r['calls (redundancy clusters)']} clusters); relabeled "
              f"{r['relabeled calls']}; NCDR {r['NCDR']:.2f} ({r['relabeling rule']}); calibration {r['calibration share']:.3f} "
              f"({r['calibration']})")

## 12 · Run record

In [ ]:
import platform

from rpy2.robjects import r as R

cache_entries = []
for sidecar in sorted(cache.glob('*.key.json'), key=lambda p: json.loads(p.read_text())['written_utc']):
    meta = json.loads(sidecar.read_text())
    if meta['stage'] in SECONDS:
        cache_entries.append({'stage': meta['stage'], 'key': meta['key'], 'written_utc': meta['written_utc'],
                              'compute seconds': round(SECONDS[meta['stage']], 1)})
CACHE_TABLE = pd.DataFrame(cache_entries).drop_duplicates('stage', keep='last').sort_values('stage').reset_index(drop=True)
display(CACHE_TABLE)
record = {
    'notebook_logic_version': NOTEBOOK_LOGIC_VERSION,
    'protocol': 'docs/results/pt-gene-model-nb.md, Notebook 69 protocol (commit 4fdcdf4)',
    'datasets': INPUT_TABLE.to_dict(), 'n_jobs': N_JOBS, 'mt_genes': MT,
    'excluded': {ds: {'library': DATA[ds]['mt_library'], 'universe': DATA[ds]['mt_universe']} for ds in DATASETS},
    'gene_model': {'module': 'pseudospace.pathway_pipelines.standard_gene_model', 'basis_df': BASIS_DF,
                   'grid_points': GRID_POINTS, 'nb_gam': nb_gam.GENE_MODEL},
    'normalized': {'denominator': 'pathway_pipelines.specimen_shape_lr', 'df2': DF2, 'squeezeVar covariate': 'log mean count',
                   'cameraPR': {'inter.gene.cor': CAMERA_COR, 'use.ranks': False}},
    'clustering': {'package': EXTERNAL, 'compare_params': repr(COMPARE_PARAMS), 'pathway_params': repr(PATHWAY_PARAMS),
                   'node_spacing': NODE_SPACING, 'floor_counts': FLOOR_COUNTS, 'zones': {ds: ZONES[ds].tolist() for ds in DATASETS},
                   'runs': RUN_TABLE.to_dict(orient='records')},
    'checks': {'a': DRIFT.to_dict(orient='records'), 'b': BALANCE.to_dict(orient='records'),
               'c': METHOD_TABLE.to_dict(orient='records'), 'd': OVERLAP.to_dict(orient='records'),
               'e': SUMMARY_E.to_dict(orient='records'), 'check_a_human_vs_mouse': CHECK_A},
    'primary_passes': PASS, 'guards': guard_log,
    'versions': {'python': platform.python_version(), 'R': str(R('R.version.string')[0]),
                 'limma': str(R('as.character(packageVersion("limma"))')[0]),
                 **{m: version(m) for m in ('numpy', 'scipy', 'pandas', 'rpy2', 'joblib', 'matplotlib', 'pseudospace-reconstruction')}},
    'stage_cache': CACHE_TABLE.to_dict(orient='records'), 'compute_seconds_total': float(sum(SECONDS.values())),
    'wall_seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print(f"Wall time {record['wall_seconds']:.0f} s; summed uncached compute time of the cached stages "
      f"{record['compute_seconds_total']:.0f} s")